# Phase 1: Implementation and Comparison of Classical Matching Mechanisms

**Authors:** Sébastien Rivière, Gwendal Déléage, Gabriel Viterbo, Rafael Sesoko

## 1. Introduction

**Context & Objectives:** Overview of the matching problem focused on assigning students to projects based on ordinal preference lists to produce a satisfactory and fair allocation. The study aims to implement and compare classical algorithms on the same dataset and explore advanced extensions.

**Algorithmic Overview:** High-level presentation of the three implemented classical mechanisms:
* Gale-Shapley (GS): Applicant-proposing and project-proposing stable baseline variants.
* Serial Dictatorship (SD): Multi-variant execution (random order, priority-based order).
* Top Trading Cycles (TTC): Core execution with explicit cycle detection.

**Data Sources:** Parallel tracking utilizing anonymized real preference lists from the latest DSAI Project matching alongside controlled synthetic preference models.

---

## 2. Notebook Configuration and Initialization

This section configures the environment and imports the necessary modules. All the algorithms and evaluation functions called here have been previously implemented and verified in the project's source files.

### 2.1. Imports

In [9]:
# External imports
import pandas as pd

# Local imports
from data import load_dataset
from mechanisms import gale_shapley, serial_dictatorship, ttc_from_matching
from metrics import rank_sum_cost, borda_welfare_cost, top_n_assigned_cost, square_rank_sum_cost, nash_welfare_cost, max_rank_cost, unassigned_students_cost
from metrics import blocking_pairs_stability_score, justified_envy_stability_score, wasteful_capacity_stability_score

### 2.2. Configuration

In [10]:
# Real data
real_data_csv_path = "../data/projects-dsai-2026-anon.csv"
real_data_project_capacity = 5

# Cost and stability functions
top_n_assigned_cost_value_of_n = 3

---

## 3. Pipeline Definition and Components

This section charts the end-to-end data flow, from raw data ingestion/generation to structured output tables.

### 3.1. Matching computation

In [11]:
algorithms = {
    "gale_shapley": gale_shapley,
    "serial_dictatorship": lambda instance: serial_dictatorship(instance, [s.id for s in instance.students]),
    "top_trading_cycles": lambda instance: ttc_from_matching(gale_shapley(instance))
}

In [12]:
def compute_matching(instance):
    return {
        algorithm_name: algorithm(instance)
        for algorithm_name, algorithm in algorithms.items()
    }

### 3.2. Cost and stability computations

In [13]:
cost_stability_functions = {
    # Cost functions
    "rank_sum_cost": rank_sum_cost,
    "borda_welfare_cost": borda_welfare_cost,
    "top_n_assigned_cost": lambda matching: top_n_assigned_cost(matching, n=top_n_assigned_cost_value_of_n),
    "square_rank_sum_cost": square_rank_sum_cost,
    "nash_welfare_cost": nash_welfare_cost,
    "max_rank_cost": max_rank_cost,
    "unassigned_students_cost": unassigned_students_cost,

    # Stability functions
    "blocking_pairs_stability_score": blocking_pairs_stability_score,
    "justified_envy_stability_score": justified_envy_stability_score,
    "wasteful_capacity_stability_score": wasteful_capacity_stability_score
}

In [14]:
def compute_cost_stability(matchings):
    return pd.DataFrame.from_dict({
        alg_name: {
            cost_name: cost_fn(matching)
            for cost_name, cost_fn in cost_stability_functions.items()
        }
        for alg_name, matching in matchings.items()
    }, orient="index")

---

## 4. Analysis

**TEMPORARY NOTE** : The TODOs below are only indications, feel free to add or change things.

### 4.1. Real data analysis

TODO : Evaluate the algorithms using the real-world dataset. Measure and compare the global cost (efficiency) and the total number of blocking pairs (stability) for each mechanism. Make visualisation for example for the rank distribution of the assignements, to visualise the number of students per projet considering their capacity, unassigned students etc.

In [15]:
instance_real = load_dataset(real_data_csv_path, real_data_project_capacity)
matching_real = compute_matching(instance_real)
results_real = compute_cost_stability(matching_real)

### 4.2. Synthetic data analysis

#### 4.2.1. Impact of Preference Homogeneity

TODO : Run simulations varying $\phi$ from 0 to 1. Measure the average cost and the number of blocking pairs for SD and TTC (especially near $\phi = 0$). Plot the results on a line graph ($\phi$ on X-axis, metrics on Y-axis).

#### 4.2.2. Scalability

TODO : Increase $N$ (students) and $M$ (projects) while keeping the capacity ratio constant. Record the empirical execution time and analyze algorithm behavior under high resource scarcity.

### 4.3. Efficiency VS Stability trade-off analysis

TODO : Generate a 2D scatter plot to map the Pareto frontier. Plot the Global Cost Score on the X-axis (Efficiency) against the Number of Blocking Pairs on the Y-axis (Instability). Use distinct markers for each algorithm to identify the best compromise.

---

## 5. Synthesis and Perspectives

### 5.1. Performance Summary

TODO

### 5.2. Transition to Phase 2

TODO